# Generate Dummy Data (Dev)

> **Status:** Exploratory — superseded by `notebooks/production/generate_prod_dummy_data.ipynb`.
> This notebook generates a small set of synthetic persons, addresses, and calendar events for use in the dev training pipeline. It targets a hard-coded 250-person set against the census ZIP dataset. The production version generates dummy data keyed to real production person IDs from the invoice snapshot.

**Outputs** (written to `data/processed/`):
- `dummy_persons.csv`
- `dummy_addresses.csv`
- `dummy_calendar_events.csv`

## Step 1: Generate Persons Data
Generates 200+ unique person records matching the `public.person` schema. This represents raw data as it would be queried from the database — no derived attributes.

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import uuid
from datetime import datetime, timedelta
import pytz

np.random.seed(42)
fake = Faker()
Faker.seed(42)

NUM_PERSONS = 250

person_records = []
for _ in range(NUM_PERSONS):
    person_id = str(uuid.uuid4())
    org_id = str(uuid.uuid4())
    dataset_id = str(uuid.uuid4())
    source_tenant_id = str(uuid.uuid4())
    source_id = str(uuid.uuid4())
    household_id = str(uuid.uuid4())

    entry_date = fake.date_time_between(start_date='-5y', end_date='-30d', tzinfo=pytz.UTC)

    person_records.append({
        'id': person_id,
        'org_id': org_id,
        'dataset_id': dataset_id,
        'source_tenant_id': source_tenant_id,
        'source_id': source_id,
        'person_pmid': fake.bothify(text='PM-######'),
        'household_id': household_id,
        'household_pmid': fake.bothify(text='HH-######'),
        'is_guardian': np.random.choice([True, False], p=[0.25, 0.75]),
        'first_name': fake.first_name(),
        'last_name': fake.last_name(),
        'preferred_name': fake.first_name() if np.random.rand() < 0.3 else None,
        'status': np.random.choice(['Active', 'Inactive', 'Prospect'], p=[0.8, 0.1, 0.1]),
        'gender': np.random.choice(['Male', 'Female', 'Other', None], p=[0.45, 0.45, 0.05, 0.05]),
        'birthdate': fake.date_of_birth(minimum_age=18, maximum_age=85).isoformat(),
        'entry_date': entry_date.isoformat(),
        'created_at': entry_date.isoformat(),
        'modified_at': fake.date_time_between(start_date=entry_date, end_date='now', tzinfo=pytz.UTC).isoformat(),
    })

persons_df = pd.DataFrame(person_records)

print(f"✅ Generated {len(persons_df)} person records")
print(f"   Guardians: {persons_df['is_guardian'].sum()} ({persons_df['is_guardian'].mean()*100:.1f}%)")
print(f"   Status distribution: {persons_df['status'].value_counts().to_dict()}")
print(f"\nPreview:")
persons_df[['id', 'first_name', 'last_name', 'is_guardian', 'birthdate', 'entry_date']].head()

✅ Generated 250 person records
   Guardians: 71 (28.4%)
   Status distribution: {np.str_('Active'): 210, np.str_('Prospect'): 22, np.str_('Inactive'): 18}

Preview:


,id,first_name,last_name,is_guardian,birthdate,entry_date
0,c375c13b-a867-4d36-bbb7-5feed52ef755,Clayton,Hall,False,1981-05-17,2024-08-01T07:02:09.089700+00:00
1,c73a705f-44b2-49ee-b8f7-a361133dc5de,Jason,Gallagher,True,1966-04-10,2024-12-16T20:51:36.642377+00:00
2,198d34a5-a072-4871-800d-4a1b5df61b51,James,Jones,False,1985-05-27,2023-02-17T13:37:56.417125+00:00
3,d7894388-84cd-4cb8-a87c-cffe7e63e95e,Chelsea,Jackson,False,1951-07-20,2026-04-14T09:49:52.963498+00:00
4,6185fa1f-6858-4a48-95d5-306ab4098e70,Nancy,Edwards,False,1985-01-03,2026-04-22T03:16:41.184522+00:00


## Step 2: Generate Addresses
Generates one `public.address` record per person. Zip codes are sampled from the census dataset (filtered to `census_match == 1`) so the census join in Step 7 always succeeds with no nulls.

In [2]:
census_path = '../../data/processed/census_zip_features_model_ready.csv'
census_df = pd.read_csv(census_path, dtype={'zip_code': str})

# Only use zip codes that have complete, valid census data
# Filter: census_match == 1, non-null income, non-negative percentages (removes sentinel values)
valid_census = census_df[
    (census_df['census_match'] == 1) &
    (census_df['median_household_income'].notna()) &
    (census_df['poverty_rate_pct'].notna()) &
    (census_df['poverty_rate_pct'] >= 0) &
    (census_df['unemployment_rate_pct'] >= 0) &
    (census_df['average_household_size'].notna()) &
    (census_df['average_household_size'] > 0)
].copy()

valid_zip_codes = valid_census['zip_code'].tolist()

print(f"Loaded {len(census_df)} zip codes total")
print(f"Valid zip codes with complete census data: {len(valid_zip_codes)}")

# Generate one address per person, sampling from valid zip codes
address_records = []
for _, person in persons_df.iterrows():
    zip_code = np.random.choice(valid_zip_codes)
    address_records.append({
        'id': str(uuid.uuid4()),
        'org_id': person['org_id'],
        'dataset_id': person['dataset_id'],
        'source_tenant_id': person['source_tenant_id'],
        'source_id': str(uuid.uuid4()),
        'person_id': person['id'],
        'address_lines': fake.street_address(),
        'city': fake.city(),
        'state': fake.state_abbr(),
        'postal_code': zip_code,
        'created_at': person['created_at'],
        'modified_at': person['modified_at'],
    })

addresses_df = pd.DataFrame(address_records)

print(f"\n✅ Generated {len(addresses_df)} address records (1 per person)")
print(f"   Unique zip codes used: {addresses_df['postal_code'].nunique()}")
print(f"\nPreview:")
addresses_df[['person_id', 'city', 'state', 'postal_code']].head()

Loaded 37963 zip codes total
Valid zip codes with complete census data: 30506

✅ Generated 250 address records (1 per person)
   Unique zip codes used: 249

Preview:


,person_id,city,state,postal_code
0,c375c13b-a867-4d36-bbb7-5feed52ef755,East Amandachester,SD,75831
1,c73a705f-44b2-49ee-b8f7-a361133dc5de,East Lauraside,CA,63645
2,198d34a5-a072-4871-800d-4a1b5df61b51,New Adamland,WV,05055
3,d7894388-84cd-4cb8-a87c-cffe7e63e95e,Catherineport,WV,39827
4,6185fa1f-6858-4a48-95d5-306ab4098e70,New Nancyberg,DE,77011


## Step 3: Generate Invoices
Generates invoice records matching the `payments.invoices` schema using the correct production `InvoiceStatus` enum values:

| Status | Value |
|---|---|
| PAID | 4 |
| PARTIALLY_PAID | 5 |
| UNPAID | 6 |
| SCHEDULED | 7 |
| CANCELED | 8 |
| PENDING | 9 |

`PaymentOrigin`: LOCATION=1, PAYMENT_PORTAL=2, TERMINAL=3, LOCATION_PORTAL=4, MOBILE_TAP_TO_PAY=5, PAYMENT_PLAN=6

In [3]:
NUM_INVOICES = 4000

# InvoiceStatus enum (from db/payments/status-enums.md)
INVOICE_STATUS = {
    'PAID': 4,
    'PARTIALLY_PAID': 5,
    'UNPAID': 6,
    'SCHEDULED': 7,
    'CANCELED': 8,
    'PENDING': 9,
}

# PaymentOrigin enum
PAYMENT_ORIGIN = {
    'LOCATION': 1,
    'PAYMENT_PORTAL': 2,
    'TERMINAL': 3,
    'LOCATION_PORTAL': 4,
    'MOBILE_TAP_TO_PAY': 5,
    'PAYMENT_PLAN': 6,
}

# Status distribution: majority paid, realistic mix
status_values = [4, 5, 6, 7, 8, 9]
status_probs = [0.55, 0.10, 0.18, 0.07, 0.05, 0.05]

# Origin distribution
origin_values = [1, 2, 3, 4, 5, 6]
origin_probs = [0.25, 0.30, 0.20, 0.10, 0.08, 0.07]

# Use a single merchant ID per "practice" for simplicity
merchant_id = str(uuid.uuid4())

invoice_records = []
for _ in range(NUM_INVOICES):
    # Pick a random person
    person = persons_df.sample(1).iloc[0]
    person_entry = pd.Timestamp(person['entry_date'])

    # Invoice created after person entry date
    created_at = fake.date_time_between(
        start_date=person_entry,
        end_date='now',
        tzinfo=pytz.UTC
    )
    # Expires 30 days after creation
    expires_at = created_at + timedelta(days=30)

    # Amount in cents (e.g. $10.00 to $2,500.00)
    amount = np.random.randint(1000, 250000)

    status = np.random.choice(status_values, p=status_probs)
    origin = np.random.choice(origin_values, p=origin_probs)

    invoice_records.append({
        'id': str(uuid.uuid4()),
        'merchantid': merchant_id,
        'personid': person['id'],
        'amount': amount,
        'createdat': created_at.isoformat(),
        'expiresat': expires_at.isoformat(),
        'status': int(status),
        'origin': int(origin),
        'hasattachment': np.random.choice([True, False], p=[0.15, 0.85]),
        'surchargingenabled': np.random.choice([True, False], p=[0.10, 0.90]),
        'providername': np.random.choice(['Stripe', 'Justifi'], p=[0.7, 0.3]),
        'personfirstname': person['first_name'],
        'personlastname': person['last_name'],
        'uniquelink': str(uuid.uuid4())[:36],
    })

invoices_df = pd.DataFrame(invoice_records)

# Print status distribution
print("✅ Generated {:,} invoice records".format(len(invoices_df)))
print(f"   Unique persons referenced: {invoices_df['personid'].nunique()}")
print(f"\nInvoice status distribution:")
status_labels = {4: 'PAID', 5: 'PARTIALLY_PAID', 6: 'UNPAID', 7: 'SCHEDULED', 8: 'CANCELED', 9: 'PENDING'}
for status_val, count in invoices_df['status'].value_counts().sort_index().items():
    label = status_labels.get(status_val, 'UNKNOWN')
    print(f"   {label} ({status_val}): {count} ({count/len(invoices_df)*100:.1f}%)")

print(f"\nPreview:")
invoices_df[['id', 'personid', 'amount', 'createdat', 'status', 'origin', 'providername']].head()

✅ Generated 4,000 invoice records
   Unique persons referenced: 250

Invoice status distribution:
   PAID (4): 2183 (54.6%)
   PARTIALLY_PAID (5): 412 (10.3%)
   UNPAID (6): 739 (18.5%)
   SCHEDULED (7): 276 (6.9%)
   CANCELED (8): 201 (5.0%)
   PENDING (9): 189 (4.7%)

Preview:


,id,personid,amount,createdat,status,origin,providername
0,2bc57502-7941-4022-a5bc-d72f19a2fcb2,be047312-e14a-4dc2-b757-2c13385a7051,10078,2026-03-25T20:02:25.629974+00:00,6,6,Stripe
1,f3a3f59d-7f3e-4cb4-b5c3-a79f5664513b,354b7f4a-2ca0-40aa-8b24-2df67f039756,204532,2025-09-17T10:40:07.548126+00:00,6,3,Stripe
2,a642c89c-4e83-4157-a82a-6b8594e44575,61917d19-c507-483c-8d82-0d4f18580b02,125982,2025-05-16T20:14:41.536368+00:00,6,2,Justifi
3,03a62775-bcd4-4e35-8d1e-c0d6cd64a014,ff081c8d-1d5e-478b-8448-a998c0863cd2,239340,2026-02-23T06:48:34.071434+00:00,6,4,Stripe
4,c75b4a4f-3722-4bc1-bd41-9df1568deb97,7bb0fd61-f112-4422-9af1-fe8f229ada53,161381,2024-01-17T00:48:39.402721+00:00,6,2,Justifi


## Step 4: Generate Payment Logs + Invoice Payments
For invoices with `PAID` or `PARTIALLY_PAID` status, generates `payments.payment_log` records and links them via `payments.invoice_payments`. `payment_log.recordedat` records when the payment was actually made — used in Step 6 to derive `average_days_to_pay`.

In [4]:
payment_log_records = []
invoice_payment_records = []

# Only generate payment logs for PAID (4) and PARTIALLY_PAID (5) invoices
paid_invoices = invoices_df[invoices_df['status'].isin([4, 5])].copy()

for _, invoice in paid_invoices.iterrows():
    invoice_created = pd.Timestamp(invoice['createdat'])

    # Determine number of payments for this invoice
    # PAID: usually 1 payment, sometimes 2 (split payments)
    # PARTIALLY_PAID: 1 payment that doesn't cover full amount
    if invoice['status'] == 4:
        num_payments = np.random.choice([1, 2], p=[0.85, 0.15])
    else:
        num_payments = 1

    remaining_amount = invoice['amount']

    for i in range(num_payments):
        payment_id = str(uuid.uuid4())

        # Payment happens between 1 and 90 days after invoice creation
        # Use a right-skewed distribution: most pay quickly, some take longer
        days_to_pay = max(1, int(np.random.exponential(scale=12) + 1))
        days_to_pay = min(days_to_pay, 90)  # cap at 90 days

        recorded_at = invoice_created + timedelta(
            days=days_to_pay,
            hours=np.random.randint(8, 20),
            minutes=np.random.randint(0, 60)
        )

        # Payment amount
        if invoice['status'] == 5:  # PARTIALLY_PAID
            payment_amount = int(remaining_amount * np.random.uniform(0.3, 0.7))
        elif num_payments > 1 and i < num_payments - 1:
            payment_amount = int(remaining_amount * np.random.uniform(0.4, 0.6))
        else:
            payment_amount = remaining_amount

        remaining_amount -= payment_amount

        payment_log_records.append({
            'id': payment_id,
            'merchantid': invoice['merchantid'],
            'amount': payment_amount,
            'paymentstatus': 1,  # successful
            'submittedat': recorded_at.isoformat(),
            'createdat': recorded_at.isoformat(),
            'recordedat': recorded_at.isoformat(),
            'processorid': str(uuid.uuid4()),
            'paymenttype': np.random.choice([0, 1, 2]),  # card types
            'origin': invoice['origin'],
            'processortype': 'stripe' if invoice['providername'] == 'Stripe' else 'justifi',
        })

        invoice_payment_records.append({
            'invoiceid': invoice['id'],
            'paymentid': payment_id,
        })

payment_log_df = pd.DataFrame(payment_log_records)
invoice_payments_df = pd.DataFrame(invoice_payment_records)

print(f"✅ Generated {len(payment_log_df)} payment_log records")
print(f"   Linked via {len(invoice_payments_df)} invoice_payments records")
print(f"   For {len(paid_invoices)} paid/partially-paid invoices")
print(f"\nPayment log preview:")
print(payment_log_df[['id', 'merchantid', 'amount', 'recordedat', 'processortype']].head())
print(f"\nInvoice payments preview:")
print(invoice_payments_df.head())

✅ Generated 2913 payment_log records
   Linked via 2913 invoice_payments records
   For 2595 paid/partially-paid invoices

Payment log preview:
                                     id                            merchantid  \
0  4dec341a-c6c4-4cd1-8392-fb123338b3ba  724882b5-630c-42bb-9088-22aaa1115a54   
1  5f230999-5cc6-4ddb-af00-15d0a677a765  724882b5-630c-42bb-9088-22aaa1115a54   
2  57618b1b-7add-4ea5-abaa-30b1ac9b4f32  724882b5-630c-42bb-9088-22aaa1115a54   
3  088289df-f6a8-4a24-9f58-781d6f8e6d59  724882b5-630c-42bb-9088-22aaa1115a54   
4  3fda44ba-e2c9-473f-a455-cb111fea6725  724882b5-630c-42bb-9088-22aaa1115a54   

   amount                        recordedat processortype  
0  217823  2025-12-08T16:59:47.436232+00:00       justifi  
1  184965  2023-10-13T10:26:19.619922+00:00        stripe  
2   52012  2025-06-06T17:17:25.100078+00:00        stripe  
3  135174  2025-11-25T19:56:35.888503+00:00       justifi  
4  113134  2025-11-25T17:44:35.888503+00:00       justifi  

Invoice 

## Step 5: Generate Calendar Events
Generates `scheduler.calendars_events` records for each person. Each person is assigned a reliability tendency that influences their `attendee_status` distribution. Used in Step 6 to derive `appointment_reliability_score`.

`attendee_status` values: `CONFIRMED`, `COMPLETED`, `CANCELED`, `NO_SHOW`, `UNCONFIRMED`

In [5]:
ATTENDEE_STATUSES = ['CONFIRMED', 'COMPLETED', 'CANCELED', 'NO_SHOW', 'UNCONFIRMED']

calendar_event_records = []

location_id = str(uuid.uuid4())
source_tenant_id = str(uuid.uuid4())
calendar_id = str(uuid.uuid4())

for _, person in persons_df.iterrows():
    person_entry = pd.Timestamp(person['entry_date'])

    # Number of appointments per person: between 2 and 40, skewed toward 5-15
    num_appointments = max(2, int(np.random.gamma(shape=4, scale=3)))
    num_appointments = min(num_appointments, 40)

    # Each person has a reliability tendency (used to weight status probabilities)
    # Some people are highly reliable, others tend to miss appointments
    reliability_tendency = np.random.beta(5, 2)  # skewed reliable

    for _ in range(num_appointments):
        # Appointment date: between person entry and now
        apt_date = fake.date_time_between(
            start_date=person_entry,
            end_date='now',
            tzinfo=pytz.UTC
        )

        # Status influenced by person's reliability tendency
        if reliability_tendency > 0.7:
            status_probs = [0.05, 0.75, 0.08, 0.02, 0.10]  # mostly completed
        elif reliability_tendency > 0.4:
            status_probs = [0.08, 0.55, 0.15, 0.10, 0.12]  # moderate
        else:
            status_probs = [0.10, 0.35, 0.20, 0.20, 0.15]  # unreliable

        attendee_status = np.random.choice(ATTENDEE_STATUSES, p=status_probs)

        # Appointment duration: 30-90 minutes
        duration_minutes = int(np.random.choice([30, 45, 60, 90], p=[0.3, 0.35, 0.25, 0.1]))
        start_time = apt_date
        end_time = apt_date + timedelta(minutes=duration_minutes)

        calendar_event_records.append({
            'id': str(uuid.uuid4()),
            'location_id': location_id,
            'source_tenant_id': source_tenant_id,
            'organizer_calendar_id': calendar_id,
            'organizer_id': str(uuid.uuid4()),
            'location_type': np.random.choice(['operatory', 'room'], p=[0.8, 0.2]),
            'type': 'APPOINTMENT',
            'attendee_id': person['id'],
            'attendee_status': attendee_status,
            'title': np.random.choice(['Cleaning', 'Check-up', 'Filling', 'Crown', 'Consultation', 'X-Ray']),
            'start_date': start_time.date().isoformat(),
            'start_time': start_time.isoformat(),
            'end_date': end_time.date().isoformat(),
            'end_time': end_time.isoformat(),
            'reference_id': str(uuid.uuid4()),
            'is_integrated': False,
            'recurring': False,
            'created_at': start_time.isoformat(),
        })

calendar_events_df = pd.DataFrame(calendar_event_records)

print(f"✅ Generated {len(calendar_events_df)} calendar event records")
print(f"   Unique attendees: {calendar_events_df['attendee_id'].nunique()}")
print(f"   Avg appointments per person: {len(calendar_events_df) / calendar_events_df['attendee_id'].nunique():.1f}")
print(f"\nAttendee status distribution:")
status_counts = calendar_events_df['attendee_status'].value_counts()
for status, count in status_counts.items():
    print(f"   {status}: {count} ({count/len(calendar_events_df)*100:.1f}%)")
print(f"\nPreview:")
calendar_events_df[['attendee_id', 'attendee_status', 'title', 'start_date']].head()

✅ Generated 3017 calendar event records
   Unique attendees: 250
   Avg appointments per person: 12.1

Attendee status distribution:
   COMPLETED: 2045 (67.8%)
   CANCELED: 351 (11.6%)
   UNCONFIRMED: 310 (10.3%)
   CONFIRMED: 172 (5.7%)
   NO_SHOW: 139 (4.6%)

Preview:


,attendee_id,attendee_status,title,start_date
0,c375c13b-a867-4d36-bbb7-5feed52ef755,CONFIRMED,Check-up,2025-08-10
1,c375c13b-a867-4d36-bbb7-5feed52ef755,COMPLETED,Crown,2025-12-17
2,c375c13b-a867-4d36-bbb7-5feed52ef755,COMPLETED,Cleaning,2025-01-30
3,c375c13b-a867-4d36-bbb7-5feed52ef755,COMPLETED,Cleaning,2025-07-15
4,c375c13b-a867-4d36-bbb7-5feed52ef755,COMPLETED,Consultation,2024-12-22


## Step 6: Save Raw Outputs
Saves all generated raw DataFrames to `data/processed/`.

| File | Description |
|---|---|
| `dummy_persons.csv` | `public.person` records |
| `dummy_addresses.csv` | `public.address` records |
| `dummy_invoices.csv` | `payments.invoices` records |
| `dummy_payment_log.csv` | `payments.payment_log` records |
| `dummy_invoice_payments.csv` | `payments.invoice_payments` linking records |
| `dummy_calendar_events.csv` | `scheduler.calendars_events` records |

In [6]:
import os

output_dir = '../../data/processed'
os.makedirs(output_dir, exist_ok=True)

# Save persons
persons_output = os.path.join(output_dir, 'dummy_persons.csv')
persons_df.to_csv(persons_output, index=False)
print(f"Saved persons: {persons_output} ({len(persons_df)} rows)")

# Save addresses
addresses_output = os.path.join(output_dir, 'dummy_addresses.csv')
addresses_df.to_csv(addresses_output, index=False)
print(f"Saved addresses: {addresses_output} ({len(addresses_df)} rows)")

# Save invoices
invoices_output = os.path.join(output_dir, 'dummy_invoices.csv')
invoices_df.to_csv(invoices_output, index=False)
print(f"Saved invoices: {invoices_output} ({len(invoices_df)} rows)")

# Save payment logs
payment_log_output = os.path.join(output_dir, 'dummy_payment_log.csv')
payment_log_df.to_csv(payment_log_output, index=False)
print(f"Saved payment_log: {payment_log_output} ({len(payment_log_df)} rows)")

# Save invoice_payments
invoice_payments_output = os.path.join(output_dir, 'dummy_invoice_payments.csv')
invoice_payments_df.to_csv(invoice_payments_output, index=False)
print(f"Saved invoice_payments: {invoice_payments_output} ({len(invoice_payments_df)} rows)")

# Save calendar events
calendar_events_output = os.path.join(output_dir, 'dummy_calendar_events.csv')
calendar_events_df.to_csv(calendar_events_output, index=False)
print(f"Saved calendar_events: {calendar_events_output} ({len(calendar_events_df)} rows)")

print(f"\n✅ All raw data saved to {os.path.abspath(output_dir)}")

Saved persons: ../../data/processed/dummy_persons.csv (250 rows)
Saved addresses: ../../data/processed/dummy_addresses.csv (250 rows)
Saved invoices: ../../data/processed/dummy_invoices.csv (4000 rows)
Saved payment_log: ../../data/processed/dummy_payment_log.csv (2913 rows)
Saved invoice_payments: ../../data/processed/dummy_invoice_payments.csv (2913 rows)
Saved calendar_events: ../../data/processed/dummy_calendar_events.csv (3017 rows)

✅ All raw data saved to /Users/pablotrujillo/Development/hackathon/will-they-pay/data/processed
